# Workshop-Notebook: Grundlagen der KI-Agenten & Werkzeugnutzung

Dieses Notebook zeigt Ihnen Schritt für Schritt, wie Sie ein Sprachmodell (**Gemini**) mit einem **Werkzeug (Tool)** verbinden und eine einfache **ReAct-Schleife (Orchestrierung)** aufbauen.

**Event:** Digitaltag 2026 | *Mittelstand-Digital Zentrum Spreeland*

### Schritt 1: Installation & Setup
Zuerst installieren wir das offizielle Google GenAI SDK und konfigurieren unseren API-Schlüssel.

In [ ]:
# Installation des SDKs
!pip install google-genai

In [ ]:
import os
from google import genai
from google.genai import types
from google.genai.errors import APIError

# Setzen Sie Ihren Gemini API-Key ein (holen Sie sich einen kostenlosen Key unter https://aistudio.google.com/)
# os.environ["GEMINI_API_KEY"] = "IHR_API_KEY"

### Schritt 2: Werkzeug definieren (Tool)
Wir definieren eine einfache Python-Funktion, die aktuelle Daten abfragt (z.B. Wetter). Diese Funktion wird dem Sprachmodell als Werkzeug übergeben.

In [ ]:
def get_wetter_cottbus(datum: str) -> str:
    """Fragt das aktuelle Wetter für Cottbus an einem bestimmten Datum ab.
    
    Args:
        datum: Das gewünschte Datum im Format JJJJ-MM-TT
    """
    # In der Praxis würde hier eine echte Wetter-API aufgerufen werden
    if datum == "2026-06-26":
        return "Cottbus: 24 Grad, sonnig, Wind aus Nord-Ost, Regenwahrscheinlichkeit 5%."
    else:
        return f"Für das Datum {datum} liegen leider keine Wetterdaten vor. Standardvorhersage: Leicht bewölkt, 18 Grad."

### Schritt 3: Modell mit Werkzeug verbinden (Tool Binding)
Wir initialisieren den Gemini-Client und übergeben unsere Python-Funktion als Liste von Tools.

In [ ]:
# Initialisierung des GenAI Clients
client = genai.Client()

# Wir nutzen gemini-2.5-flash als unser schnelles Entscheidungsmodell
model_id = 'gemini-2.5-flash'

response = client.models.generate_content(
    model=model_id,
    contents="Brauche ich am 26. Juni 2026 in Cottbus eine Regenjacke?",
    config=types.GenerateContentConfig(
        # Hier registrieren wir unsere Funktion als Werkzeug
        tools=[get_wetter_cottbus],
        temperature=0
    )
)

# Das Modell entscheidet, ob es das Werkzeug aufrufen muss. 
# Wenn ja, liefert es uns einen Funktionsaufruf (Function Call) zurück:
print("Funktionsaufrufe des Modells:", response.function_calls)

### Schritt 4: Die Orchestrierungsschleife (ReAct Loop)
Wenn das Modell entscheidet, das Werkzeug aufzurufen, müssen wir den Funktionsaufruf abfangen, die Python-Funktion ausführen und das Ergebnis zurück an das Modell senden, damit dieses die endgültige Antwort formuliert.

In [ ]:
# Automatischer Durchlauf
# Die SDKs unterstützen auch automatische Funktionsaufrufe, sodass die Schleife im Hintergrund läuft:
response_auto = client.models.generate_content(
    model=model_id,
    contents="Brauche ich am 26. Juni 2026 in Cottbus eine Regenjacke? Beantworte basierend auf dem Wetterbericht.",
    config=types.GenerateContentConfig(
        # Das SDK führt die Funktion automatisch im Hintergrund aus und liefert das Gesamtergebnis
        tools=[get_wetter_cottbus],
        # Ermöglicht automatische Ausführung
        automatic_function_calling=types.AutomaticFunctionCallingConfig(max_remote_calls=3)
    )
)

print("Endgültige Antwort des Agenten:\n")
print(response_auto.text)